## Execution order — IMPORTANT

The original `data_processing.md` documents the notebook order as 03 → 04 → 05.
**This is incorrect.** The actual required execution order is:

1. `snATAC_03` — merge samples and initial clustering
2. `snATAC_05` — label transfer (first pass); produces `snATAC_Lt_filt05AcinarSum.rds`
3. **`snATAC_04` (this notebook)** — manual doublet removal using promoter accessibility; reads the label-transferred object from step 2
4. `snATAC_05` — re-run the final acinar-sum filtering section on the cleaned object

This cross-dependency was discovered during reproduction (repro branch) by tracing file
inputs/outputs.  It contradicts the authors' documented order but matches the actual
file names produced and consumed.

---

## Path configuration (repro branch)

Hardcoded `/data/`, `/reference_files/`, and NFS paths replaced with `Sys.getenv()` calls
(see setup cell below).

| Variable | Default | Purpose |
|---|---|---|
| `NPOD_DATA_DIR` | `/data` | Root data directory |
| `NPOD_REF_DIR` | `/reference_files` | Reference annotation files |
| `NPOD_PROM_ACC_DIR` | `$NPOD_DATA_DIR/peak_calling/FilterATAC_byPromoterAccessibility` | Per-cell-type promoter accessibility matrix output |
| `NPOD_FRAG_FILE` | `$NPOD_DATA_DIR/merged.atac_fragments.tsv.gz` | Merged fragment file |
| `NPOD_LT_DIR` | `$NPOD_DATA_DIR/snATAC_LT_files` | Label-transfer intermediate and output files |

Since CFTR in gencode appears off from where we are seeing true peaks (those in the ductal population), we have decided to use Coordinates used in UCSC for all marker genes that appear to differ from gencode

Marker list: "REG1A" "REG1B" "PRSS1" "INS" "GCG" "CFTR" "CTRB2" "PRSS2" "C1QC" 

- REG1A: ENST00000485184.1
    - chr2:79120835-79123091
    - promoter region: 2:79118835-79120835
- REG1B: ENST00000305089.8
    - chr2:79085023-79087993
    - promoter region: 2:79087993-79089993
- PRSS1: ENST00000492062.1
    - chr7:142750665-142753069
    - promoter region: 7:142748665-142750665
- INS: ENST00000250971.7
    - chr11:2159779-2161221
    - promoter region: 11:2161221-2163221
- GCG: ENST00000418842.7
    - chr2:162142882-162152247
    - promoter region: 2:162152247-162154247
- CFTR: ENST00000699597.1 
    - chr7:117479988-117559913
    - promoter region: 7:117477988-117479988
- CTRB2: ENST00000303037.13
    - chr16:75204103-75207161
    - promoter region: 16:75207161-75209161
- PRSS2: ENST00000539842.6
    - chr7:142770970-142774560	
    - promoter region: 7:142768970-142770970
- C1QC: ENST00000695751.1
    - chr1:22643626-22648110
    - promoter region: 1:22641626-22643626

Get gencode file from https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_41/gencode.v41.annotation.gtf.gz

```bash
#### create Promoter file
zcat gencode.v41.annotation.gtf.gz |awk '$3=="transcript"'| grep -f UCSC_ids.txt | awk 'BEGIN{FS=OFS="\t"} {if($7=="+") {split($9,a,"; "); print $1,$4,$5,a[4]} else if($7=="-") {split($9,a,"; "); print $1,$4,$5,a[4]}}' | sed 's/"\|gene_name //g' |sort -k1,1 -k2,2n > UCSC_doubleCheckCoords.txt

zcat gencode.v41.annotation.gtf.gz |awk '$3=="transcript"'| grep -f UCSC_ids.txt | awk 'BEGIN{FS=OFS="\t"} {if($7=="+") {split($9,a,"; "); print $1,$4-2000,$4,a[4]} else if($7=="-") {split($9,a,"; "); print $1,$5,$5+2000,a[4]}}' | sed 's/"\|gene_name //g' |sort -k1,1 -k2,2n | awk '!uniq[$3]++' > jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors.bed

### mod file for different analysis
awk -v OFS='\t' ' { print $1 "-" $2 "-"  $3, $4} ' jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors.bed | awk '{print substr($0,4)}' > jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors_geneNames.txt

awk -v OFS='\t' ' { print $1 ":" $2 "-"  $3} ' jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors.bed | awk '{print substr($0,4)}' > jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors_NoNames.txt
```

```bash
##### make count matrices by celltype using TSS promoter list
#!/bin/bash
Pyscript_fp=/scripts/Josh_10XPipeline_withPeaks_justLFM_UCSCcoordsShortList.py
samples=(Acinar_1_2_6 Acinar_3 Acinar_5 Acinar_4 Activated_Stellate Endothelial Schwann Alpha Delta Beta Macrophage Tcells Mast Quiescent_Stellate LymphEndo Bcells MUC5b_Ductal Ductal)
N=5
for sample in ${samples[@]}; do
        ((i=i%N)); ((i++==0)) && wait
        (
                outs_dir=/data/peak_calling/FilterATAC_byPromoterAccessibility/${sample}/
                mkdir $outs_dir
                keep_dir=/data/peak_calling/${sample}

                tagAlign=/data/peak_calling/nPOD_spiltTagAlign.${sample}

                /usr/bin/python3 $Pyscript_fp  -o $outs_dir -k ${keep_dir}_barcodes.txt -n ${sample} -a ${tagAlign}.filt.rmdup.2022-12-20.broad.tagAlign.gz -t 24 -m 2 > ${outs_dir}log_file.txt
        ) &
done

```

# Seurat work

In [ ]:
suppressMessages(library(Matrix))
suppressMessages(library(tibble))
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
suppressMessages(library(sctransform))
suppressMessages(library(scater))
suppressMessages(library(reticulate))
suppressMessages(library('Biobase'))
suppressMessages(library(pheatmap))
suppressMessages(library(gplots))
suppressMessages(library(BiocParallel))
suppressMessages(library(tictoc))
suppressMessages(library(cowplot))
suppressMessages(library(GenomeInfoDb))
# EnsDb.Hsapiens.v75 removed (repro branch): confirmed unused — not called anywhere
# in this notebook.  v75 is the hg19/GRCh37 annotation; this analysis uses hg38.
# Removing it avoids installing an incompatible genome build's annotation package.
# EnsDb.Mmusculus.v79 removed for the same reason (mouse annotation, human study).

In [ ]:
DATA_DIR     <- Sys.getenv('NPOD_DATA_DIR',      '/data')
REF_DIR      <- Sys.getenv('NPOD_REF_DIR',       '/reference_files')
PROM_ACC_DIR <- Sys.getenv('NPOD_PROM_ACC_DIR',  file.path(DATA_DIR, 'peak_calling', 'FilterATAC_byPromoterAccessibility'))
FRAG_FILE    <- Sys.getenv('NPOD_FRAG_FILE',     file.path(DATA_DIR, 'merged.atac_fragments.tsv.gz'))
LT_DIR       <- Sys.getenv('NPOD_LT_DIR',        file.path(DATA_DIR, 'snATAC_LT_files'))

In [ ]:
#function which takes in a list of long format atac_fragment dfs with 
#sample names (df), an overall windows file (windows) and then makes these 
#into sparse matrices and merges them together

#modified to take in the hvws set and use those... will still check if 
#there's any missing windows and add those, so only a few changes!

merge_sparse_matrices_hvws <- function(dfs, windows){
    samples <- names(dfs)
    for (sample in samples){
        #get missing windows list for this sample
        print(paste(sample,Sys.time(),sep=': '))
        df <- dfs[[sample]]
        mis_windows <- windows[!windows %in% levels(df$V1)]
        
        #make sure there are missing_windows
        if (length(mis_windows) > 0){
            print('Adding missing windows')
            #create a new long format matrix (sm) with the missing windows added as 0 counts
            filler_bc <- as.character(df$V2[[1]])
            print(paste("Using filler BC:",filler_bc,sep=" "))
            new_rows <- cbind(as.data.frame(mis_windows),
                              as.data.frame(rep(filler_bc),length(mis_windows)),
                              as.data.frame(rep(0,length(mis_windows))))
            colnames(new_rows) <- c("V1","V2","V3")
            lfm <- rbind(df,new_rows)
        #if there aren't, set lfm to df
        } else {
            print('No windows were missing')
            lfm <- df
        }
        
        #cut down lfm to just be the hvws (windows)
        lfm_cut <- lfm[lfm$V1 %in% windows,]
        #set the levels of the lfm based on the desired bc order and reorder V1
        lfm_cut$V1 <- factor(lfm_cut$V1, levels=windows)
        lfm2 <- lfm_cut[order(lfm_cut$V1),]
        lfm2$V2 <- as.factor(lfm2$V2)
        
        if (sample == samples[1]){
            #if first sample, will make the overall sparse matrix 
            overall_sm <- with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(overall_sm))
            
        } else {
            #lfm2 <- lfm2[lfm2$V2 %in% good,]

            #otherwise, convert into a sparse matrix and add to the overall one
            sm = with(lfm2,sparseMatrix(i=as.numeric(V1), j=as.numeric(V2), x=V3, dimnames=list(levels(V1), levels(V2))))
            print(dim(sm))
            overall_sm = cbind(overall_sm, sm) 
        }
    }
    return(overall_sm)
}

In [ ]:
### read in original snATAC object
# NOTE: this file is produced by snATAC_05 (label transfer).
# See execution order note at the top of this notebook.
atac_obj <- readRDS(file.path(LT_DIR, 'snATAC_Lt_filt05AcinarSum.rds'))
atac_obj

In [ ]:
atacs_OG <- list()
atacs_FINAL <- list()
samples <- c('Acinar_1_2_6',  'Acinar_3' , 'Acinar_4' , 'Acinar_5',
             'Activated_Stellate' , 'Alpha' , 'Bcells' , 'Beta' ,
             'Delta' , 'Ductal',  'Endothelial' , 'LymphEndo'  ,'Macrophage',
             'Mast' ,  'Quiescent_Stellate'  , 'Tcells')
for (sample in samples) {
    wd <- file.path(PROM_ACC_DIR, sample)
    atacs_OG[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    atacs_FINAL[[sample]] <- read.table(file.path(wd, sprintf('%s.long_fmt_mtx.txt.gz',sample)), sep='\t', header=FALSE, stringsAsFactors=FALSE)
    atacs_FINAL[[sample]]$V1 <- as.factor(atacs_OG[[sample]]$V2)
    atacs_FINAL[[sample]]$V2 <- as.factor(atacs_OG[[sample]]$V1)
    atacs_OG[[sample]] <- NULL
}
atac_mod <- atacs_FINAL

In [ ]:
atac_mod$Acinar_1_2_6$V2 <- paste0(atac_mod$Acinar_1_2_6$V2, "-1")
atac_mod$Acinar_3$V2 <- paste0(atac_mod$Acinar_3$V2, "-1")
atac_mod$Acinar_4$V2 <- paste0(atac_mod$Acinar_4$V2, "-1")
atac_mod$Acinar_5$V2 <- paste0(atac_mod$Acinar_5$V2, "-1")
atac_mod$Activated_Stellate$V2 <- paste0(atac_mod$Activated_Stellate$V2, "-1")
atac_mod$Alpha$V2 <- paste0(atac_mod$Alpha$V2, "-1")
atac_mod$Bcells$V2 <- paste0(atac_mod$Bcells$V2, "-1")
atac_mod$Beta$V2 <- paste0(atac_mod$Beta$V2, "-1")
atac_mod$Delta$V2 <- paste0(atac_mod$Delta$V2, "-1")
atac_mod$Ductal$V2 <- paste0(atac_mod$Ductal$V2, "-1")
atac_mod$Endothelial$V2 <- paste0(atac_mod$Endothelial$V2, "-1")
atac_mod$LymphEndo$V2 <- paste0(atac_mod$LymphEndo$V2, "-1")
atac_mod$Macrophage$V2 <- paste0(atac_mod$Macrophage$V2, "-1")
atac_mod$Mast$V2 <- paste0(atac_mod$Mast$V2, "-1")
atac_mod$Quiescent_Stellate$V2 <- paste0(atac_mod$Quiescent_Stellate$V2, "-1")
atac_mod$Tcells$V2 <- paste0(atac_mod$Tcells$V2, "-1")


In [ ]:
hvw_fp2 <- file.path(REF_DIR, 'jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors_NoNames.txt')
hvws2 <- scan(hvw_fp2, what="", sep="\n")
print(head(hvws2))
hvws_fin2 <- sort(hvws2)
print(head(hvws_fin2))

In [ ]:
overall_sm2 <- merge_sparse_matrices_hvws(atac_mod,hvws_fin2)
dim(overall_sm2)
head(overall_sm2)

#check if the windows of overall_sm are sorted (they should be)
sorted_windows2 <- sort(row.names(overall_sm2))
table(sorted_windows2 == row.names(overall_sm2))

In [ ]:
atac <- overall_sm2

In [ ]:
# Portability fix (repro branch): original path was an NFS absolute path
# (/nfs/lab/rlmelton/npod/...) that does not exist outside the authors' cluster.
# Replaced with FRAG_FILE env var (default: $NPOD_DATA_DIR/merged.atac_fragments.tsv.gz).
atac_assay <- CreateChromatinAssay(counts=atac, sep=c(':', '-'), genome='hg38', fragments=FRAG_FILE, min.cells=0, min.features=-1)

In [ ]:
UCSC_subFeatures <- CreateSeuratObject(
  counts = atac_assay,
  assay = "peaks"
)

In [ ]:
UCSC_subFeatures <- AddMetaData(UCSC_subFeatures, atac_obj@meta.data)


In [ ]:
gene_names <- read.table(file.path(REF_DIR, 'jan17_2kbUpstream_hg38Promoters_gencode_v41_UCSCcoors_geneNames.txt'))
x <- rownames(UCSC_subFeatures@assays$peaks@counts)
rownames(UCSC_subFeatures@assays$peaks@counts) <- gene_names$V2[match(x, gene_names$V1)]
head(UCSC_subFeatures@assays$peaks@counts)

In [ ]:
#### get count tables 
print("REG1A")
table((UCSC_subFeatures@assays$peaks@counts['REG1A',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("REG1B")
table((UCSC_subFeatures@assays$peaks@counts['REG1B',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("PRSS1")
table((UCSC_subFeatures@assays$peaks@counts['PRSS1',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)

print("INS")
table((UCSC_subFeatures@assays$peaks@counts['INS',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("GCG")
table((UCSC_subFeatures@assays$peaks@counts['GCG',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("CFTR")
table((UCSC_subFeatures@assays$peaks@counts['CFTR',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)


print("CTRB2")
table((UCSC_subFeatures@assays$peaks@counts['CTRB2',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("PRSS2")
table((UCSC_subFeatures@assays$peaks@counts['PRSS2',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)
print("C1QC")
table((UCSC_subFeatures@assays$peaks@counts['C1QC',]>0), UCSC_subFeatures@meta.data$FinalSubtypes)

In [ ]:
#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_acinar_clusters = c('Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Beta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_acinar_clusters & (UCSC_subFeatures@assays$peaks@counts['REG1A',]>0), c()], 'UCSC_acinar_REG1A.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_acinar_clusters = c('Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Beta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_acinar_clusters & (UCSC_subFeatures@assays$peaks@counts['REG1B',]>0), c()], 'UCSC_acinar_REG1B.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_acinar_clusters = c('Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Beta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.tableUCSC_subFeatures
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_acinar_clusters & (UCSC_subFeatures@assays$peaks@counts['PRSS1',]>0), c()], 'UCSC_acinar_PRSS1.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_beta_clusters = c('Acinar_1_2_6','Acinar_3','Acinar_5','Acinar_4','Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_beta_clusters & (UCSC_subFeatures@assays$peaks@counts['INS',]>0), c()], 'UCSC_beta_INS.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_alpha_clusters = c('Acinar_1_2_6','Acinar_3','Acinar_5','Acinar_4','Ductal','Activated_Stellate','Endothelial','Schwann','Beta','Delta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_alpha_clusters & (UCSC_subFeatures@assays$peaks@counts['GCG',]>0), c()], 'UCSC_alpha_GCG.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_ductal_clusters = c('Acinar_1_2_6','Acinar_3','Acinar_5','Acinar_4','Beta','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_ductal_clusters & (UCSC_subFeatures@assays$peaks@counts['CFTR',]>0), c()], 'UCSC_ductal_CFTR.promFilt',col.names=FALSE, quote=FALSE)


#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_acinar_clusters = c('Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Beta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_acinar_clusters & (UCSC_subFeatures@assays$peaks@counts['CTRB2',]>0), c()], 'UCSC_acinar_CTRB2.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_acinar_clusters = c('Ductal','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Beta','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_acinar_clusters & (UCSC_subFeatures@assays$peaks@counts['PRSS2',]>0), c()], 'UCSC_acinar_PRSS2.promFilt',col.names=FALSE, quote=FALSE)

#table((atac_obj@assays$PeaksTSS_011223@counts['REG1A',]>0), atac_obj@meta.data$FinalSubtypes)
non_mac_clusters = c('Acinar_1_2_6','Acinar_3','Acinar_5','Acinar_4','Beta','Activated_Stellate','Endothelial','Schwann','Alpha','Delta','Ductal','Tcells','Mast','Quiescent_Stellate','LymphEndo','Bcells','MUC5b_Ductal')
#write.table(
write.table(UCSC_subFeatures[[]][UCSC_subFeatures$FinalSubtypes %in% non_mac_clusters & (UCSC_subFeatures@assays$peaks@counts['C1QC',]>0), c()], 'UCSC_mac_C1QC.promFilt',col.names=FALSE, quote=FALSE)


In [ ]:
atac_og <- readRDS(file.path(LT_DIR, 'snATAC_Lt_filt05AcinarSum.rds'))

to_remove  <- "UCSC_acinar_REG1A.promFilt"
to_remove2 <- "UCSC_acinar_REG1B.promFilt"
to_remove3 <- "UCSC_acinar_PRSS1.promFilt"
to_remove4 <- "UCSC_beta_INS.promFilt"
to_remove5 <- "UCSC_alpha_GCG.promFilt"
to_remove6 <- "UCSC_ductal_CFTR.promFilt"
to_remove7 <- "UCSC_acinar_CTRB2.promFilt"
to_remove8 <- "UCSC_acinar_PRSS2.promFilt"
to_remove9 <- "UCSC_mac_C1QC.promFilt"

remove.cells1 <- trimws(scan(to_remove,  what='', sep='\n'), which='right', whitespace=' ')
remove.cells2 <- trimws(scan(to_remove2, what='', sep='\n'), which='right', whitespace=' ')
remove.cells3 <- trimws(scan(to_remove3, what='', sep='\n'), which='right', whitespace=' ')
remove.cells4 <- trimws(scan(to_remove4, what='', sep='\n'), which='right', whitespace=' ')
remove.cells5 <- trimws(scan(to_remove5, what='', sep='\n'), which='right', whitespace=' ')
remove.cells6 <- trimws(scan(to_remove6, what='', sep='\n'), which='right', whitespace=' ')
remove.cells7 <- trimws(scan(to_remove7, what='', sep='\n'), which='right', whitespace=' ')
remove.cells8 <- trimws(scan(to_remove8, what='', sep='\n'), which='right', whitespace=' ')
remove.cells9 <- trimws(scan(to_remove9, what='', sep='\n'), which='right', whitespace=' ')

all.cells <- c(remove.cells1, remove.cells2, remove.cells3,
               remove.cells4, remove.cells5, remove.cells6,
               remove.cells7, remove.cells8, remove.cells9)
print(length(all.cells))

atac_og$remove_cells <- (Cells(atac_og) %in% all.cells)
adata_new <- subset(atac_og, subset=remove_cells==FALSE)
adata_new

In [ ]:
DefaultAssay(adata_new) <- 'ATAC'
adata_new <- RunTFIDF(adata_new)
adata_new <- FindTopFeatures(adata_new, min.cutoff='q0', verbose=FALSE)
adata_new <- RunSVD(adata_new)

hm_atac <- HarmonyMatrix(Embeddings(adata_new, reduction='lsi'),adata_new@meta.data,  c("library","sex"), do_pca=FALSE,plot_convergence = TRUE, verbose = TRUE)
adata_new[['harmony.atac']] <- CreateDimReducObject(embeddings=hm_atac, key='atac_', assay='ATAC')
adata_new <- RunUMAP(adata_new, dims=2:30, reduction='harmony.atac', reduction.name='umap.atac', reduction.key='atacUMAP_')

adata_new <- FindNeighbors(object = adata_new, reduction = 'harmony.atac', dims = 2:30)
adata_new <- FindClusters(object = adata_new, algorithm=4,resolution = 0.5,method = "igraph") 

DimPlot(object = adata_new, label = TRUE) + NoLegend()
options(repr.plot.width=10, repr.plot.height=10)
p3 <- DimPlot(adata_new, reduction='umap.atac',label=TRUE, label.size=6, repel=TRUE, raster=FALSE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC')
p3

In [ ]:
p3 <- DimPlot(adata_new, reduction='umap.atac',label=TRUE,group.by = 'FinalSubtypes', label.size=6, repel=TRUE, raster=FALSE) + ggtitle('WNN')
p3 <- p3 + xlab('UMAP 1') + ylab('UMAP 2') + ggtitle('ATAC')
p3

In [ ]:
saveRDS(adata_new, file = file.path(DATA_DIR, 'snATAC_filtPromUCSCcoords.rds'))